## Focal loss for binary classification

In [16]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class BinaryFocalLoss(nn.Module):
    def __init__(self, alpha=0.25, gamma=2.0, reduction='mean'):
        super(BinaryFocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, inputs, targets):
        # inputs: [N, 1] or [N], logits from the model
        # targets: [N, 1] or [N], ground truth labels (0 or 1)
        
        # Calculate Binary Cross Entropy with Logits for numerical stability
        p = torch.sigmoid(inputs)
        ce_loss = F.binary_cross_entropy_with_logits(inputs, targets, reduction='none')
        
        # Calculate p_t
        # p_t is p if y=1, and (1-p) if y=0
        p_t = p * targets + (1 - p) * (1 - targets)
        
        # Calculate alpha_t
        alpha_t = self.alpha * targets + (1 - self.alpha) * (1 - targets)
        
        # Focal Loss formula: alpha_t * (1 - p_t)^gamma * CE
        loss = alpha_t * ((1 - p_t) ** self.gamma) * ce_loss
        
        if self.reduction == 'mean':
            return loss.mean()
        elif self.reduction == 'sum':
            return loss.sum()
        return loss

In [17]:
import torch

# --- Setup Toy Data ---
# Logits: High positive = confident y=1, High negative = confident y=0
# Let's create: [Easy Background, Hard Background, Easy Defect, Hard Defect]
logits = torch.tensor([[-5.0], [-0.5], [5.0], [0.5]], dtype=torch.float32)
targets = torch.tensor([[0.0], [0.0], [1.0], [1.0]], dtype=torch.float32)

# Instantiate Loss
criterion_bfl = BinaryFocalLoss(alpha=0.25, gamma=2.0, reduction='none')

# Calculate Loss
loss = criterion_bfl(logits, targets)

print("--- Binary Focal Loss Analysis ---")
print(f"Easy Background (p~0.006): {loss[0].item():.6f} (Extremely low)")
print(f"Hard Background (p~0.377): {loss[1].item():.6f} (Significant)")
print(f"Easy Defect     (p~0.993): {loss[2].item():.6f} (Silenced)")
print(f"Hard Defect     (p~0.622): {loss[3].item():.6f} (Primary focus)")

--- Binary Focal Loss Analysis ---
Easy Background (p~0.006): 0.000000 (Extremely low)
Hard Background (p~0.377): 0.050680 (Significant)
Easy Defect     (p~0.993): 0.000000 (Silenced)
Hard Defect     (p~0.622): 0.016893 (Primary focus)


## Focal loss for multiclass classification

In [18]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class MulticlassFocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0, reduction='mean'):
        super(MulticlassFocalLoss, self).__init__()
        self.gamma = gamma
        self.reduction = reduction
        
        # Ensure alpha is a tensor if provided as a list
        if isinstance(alpha, list):
            self.register_buffer('alpha', torch.tensor(alpha))
        else:
            self.alpha = alpha

    def forward(self, inputs, targets):
        # inputs: [Batch, Classes] (logits)
        # targets: [Batch] (class indices)
        
        ce_loss = F.cross_entropy(inputs, targets, reduction='none')
        p = torch.exp(-ce_loss) # This is p_t (prob of the correct class)
        
        focal_loss = ((1 - p) ** self.gamma) * ce_loss
        
        if self.alpha is not None:
            # Gather the specific alpha weight for each target class in the batch
            # if alpha = [0.1, 0.5, 0.4], and targets = [0, 2], alpha_t = [0.1, 0.4]
            alpha_t = self.alpha.gather(0, targets)
            focal_loss = alpha_t * focal_loss
            
        if self.reduction == 'mean':
            return focal_loss.mean()
        return focal_loss

In [19]:
import torch
import torch.nn.functional as F

# 1. Define the Number of Classes
num_classes = 3 # 0: BG, 1: Car, 2: Pedestrian

# 2. Form the Alpha Vector 
# Let's use a constant alpha of 0.25 for all foreground objects
alpha_scalar = 0.25
alpha_vec = torch.ones(num_classes)
alpha_vec[0] = 1 - alpha_scalar # Background gets 0.75
alpha_vec[1:] = alpha_scalar    # All foregrounds get 0.25

# 3. Setup Toy Data
# We'll use your logits from before
logits_mc = torch.tensor([
    [10.0, 0.0, 0.0],  # Easy Background (y=0)
    [2.0,  8.0, 0.0],  # Easy Car (y=1)
    [1.0,  1.1, 1.2],  # Hard Pedestrian (y=2)
    [5.0,  0.0, 5.1]   # Hard/Confused Pedestrian (y=2)
], dtype=torch.float32)

targets_mc = torch.tensor([0, 1, 2, 2], dtype=torch.long)

# 4. Instantiate and Call
# Using the MulticlassFocalLoss class defined earlier
criterion = MulticlassFocalLoss(alpha=alpha_vec, gamma=2.0, reduction='none')
loss_mc = criterion(logits_mc, targets_mc)

# --- Analysis ---
classes = ["Background", "Car", "Pedestrian", "Pedestrian (Confused)"]
print("--- Results with Vectorized Constant Alpha ---")
for i, l in enumerate(loss_mc):
    print(f"Sample {i} [{classes[i]}]: Alpha used: {alpha_vec[targets_mc[i]]:.2f} | Loss: {l.item():.6f}")

--- Results with Vectorized Constant Alpha ---
Sample 0 [Background]: Alpha used: 0.75 | Loss: 0.000000
Sample 1 [Car]: Alpha used: 0.25 | Loss: 0.000000
Sample 2 [Pedestrian]: Alpha used: 0.25 | Loss: 0.100314
Sample 3 [Pedestrian (Confused)]: Alpha used: 0.25 | Loss: 0.036790
